In [34]:
pip install torch-geometric

Note: you may need to restart the kernel to use updated packages.


In [74]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import json
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils import data
import matplotlib.pyplot as plt
from collections import defaultdict

from tqdm import tqdm

In [36]:
df = torch.load(r'/kaggle/input/datasets/qwerte123/hetero-data-updated-diffemb/heterodata_object12_updated.pt', weights_only=False,map_location=torch.device('cpu'))

In [37]:
df

HeteroData(
  item={
    x=[12101, 100],
    text=[12101, 2],
    related=[12101],
    is_train=[12101],
  },
  (user, rated, item)={
    history={
      train={
        item_ID=[22363],
        item_ID_next=[22363],
        user_ID=[22363],
      },
      valid={
        item_ID=[22363, 15],
        item_ID_next=[22363],
        user_ID=[22363],
      },
      test={
        item_ID=[22363, 15],
        item_ID_next=[22363],
        user_ID=[22363],
      },
    },
  }
)

In [38]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.output_dim = output_dim
        self.encoder = nn.Sequential()
        self.dims = [input_dim] + hidden_dims + [output_dim]
        for ind, inp_dim in enumerate(self.dims[:-1]):
            out_dim = self.dims[ind+1]
            cur_enc = nn.Linear(inp_dim, out_dim, bias = False)
            self.encoder.append(cur_enc)
            if ind != len(self.dims) - 2:
                self.encoder.append(nn.ReLU())
        
    def forward(self, x):
        return self.encoder(x)

In [39]:
class Codebook(nn.Module):
    def __init__(self, codebook_size, emb_dim, beta):
        super().__init__()
        self.beta = beta
        self.emb = nn.Embedding(codebook_size, emb_dim)
        nn.init.uniform_(self.emb.weight, -1.0, 1.0)

    def forward(self, x):
        code = self.emb.weight  
        dist = (
            (x ** 2).sum(dim=1, keepdim=True)
            + (code ** 2).sum(dim=1)
            - 2 * x @ code.T
        )
        ids = dist.argmin(dim=1)
        emb = self.emb(ids)
        loss = (
            F.mse_loss(emb, x.detach()) +
            self.beta * F.mse_loss(x, emb.detach())
        )
        emb_st = x + (emb - x).detach()
        return loss, emb_st, ids


In [40]:
class RQVAE(nn.Module):
    def __init__(self, inp_size, hidden_sizes, embed_dim,n_codebooks, n_layers, beta):
        super().__init__()
        self.inp_size = inp_size
        self.hidden_sizes = hidden_sizes
        self.emb_dim = embed_dim
        self.enc = Encoder(inp_size, hidden_sizes, embed_dim)
        self.dec = Encoder(embed_dim, hidden_sizes[-1::-1], inp_size)
        self.layers = nn.ModuleList(modules=[Codebook(n_codebooks, embed_dim, beta) for _ in range(n_layers)])
    
    def forward(self,x):
        encoded = self.enc(x)
        r = encoded
        q_losses = 0
        sids = []
        e_sum = torch.zeros(size=(x.shape[0], self.emb_dim), device = 'cuda' if torch.cuda.is_available() else 'cpu')
        for layer in self.layers:
            q_loss, emb, ids = layer(r)
            r = r-emb.detach()
            q_losses+= q_loss
            e_sum += emb
            sids.append(ids)
        dec = self.dec(e_sum)
        rec_loss = F.mse_loss(x, dec) 
        loss = q_losses + rec_loss
        return {"sids": sids, 
                "q_loss": q_losses,
                "r_loss": rec_loss,
                "loss": loss}   
            
            

In [86]:
class RQVAE_modified(nn.Module):
    def __init__(self, inp_size, hidden_sizes, embed_dim, n_layers, beta, gamma):
        super().__init__()
        self.inp_size = inp_size
        self.hidden_sizes = hidden_sizes
        self.emb_dim = embed_dim
        self.enc = Encoder(inp_size, hidden_sizes, embed_dim)
        self.dec = Encoder(embed_dim, hidden_sizes[-1::-1], inp_size)
        self.layers = nn.ModuleList(modules=[Codebook(2048//(2**i), embed_dim, beta) for i in range(n_layers)])
    
    def forward(self,x, related):
        encoded = self.enc(x)
        rel_encoded = self.enc(related)
        r = encoded
        q_losses = 0
        sids = []
        e_sum = torch.zeros_like(encoded)
        level_mask = torch.randint(1, len(self.layers)+1, size=(1,)).item()
        for level, layer in enumerate(self.layers):
            mask = level < level_mask
            q_loss, emb, ids = layer(r)
            r = r-emb.detach()
            q_losses+= q_loss
            e_sum += emb * mask
            sids.append(ids)
        dec = self.dec(e_sum)
        rec_loss = F.mse_loss(x, dec) 
        sims = encoded @ rel_encoded.T
        log_probs = F.log_softmax(sims, dim=1)
        con_loss = -torch.diag(log_probs).mean()
        loss = q_losses + rec_loss + gamma * con_loss
        return {"sids": sids, 
                "q_loss": q_losses,
                "r_loss": rec_loss,
                "con_loss": con_loss,
                "loss": loss}   

In [42]:
class Data_Embeddings(Dataset):
    def __init__(self, embed, related):
        super().__init__()
        self.embed = embed
        self.related = related
        
    def __getitem__(self, index):
        rel_id = self.related[index]
        if len(rel_id) == 0:
            rel_embed = self.embed[index]
        else:
            rel_embed = self.embed[np.random.choice(rel_id)]
        return rel_embed, self.embed[index]
        
    def __len__(self):
        return self.embed.shape[0]

In [43]:
def unique_per_level(SIDs):
    uns = []
    for sid in SIDs:
        uns.append(sid.unique().numel())
    return torch.tensor(uns, device = device)

In [44]:
def entropy_per_level(all_sids, codebook_size):
    entropies = []

    for level_ids in all_sids:
        ids = torch.cat(level_ids).view(-1)
        counts = torch.bincount(ids, minlength=codebook_size).float()
        probs = counts / counts.sum()
        probs = probs[probs > 0]
        entropy = -(probs * torch.log2(probs)).sum()
        entropies.append(entropy)

    return torch.tensor(entropies, device='cpu').detach()


In [45]:
embeds = df['item'].x
related = df['item'].related
ds = Data_Embeddings(embeds, related)

In [46]:
train_rq, val_rq = data.random_split(ds, [0.9, 0.1])
print(len(val_rq))

1210


In [47]:
dl_train_rq = DataLoader(train_rq, batch_size = 1024, shuffle = True)
dl_val_rq = DataLoader(val_rq, batch_size = len(val_rq), shuffle= False)

In [48]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [87]:
lr = 0.4
n_layers = 3
beta = 0.25
gamma = 0.1
hidden_sizes = [512, 256, 128]
embed_dim = 32
codebook_num = 2048
n_ep = 3000

In [88]:
def train_model(model, n_ep, optimizer, dl_train, dl_val, n_layers, device):
    history = {
    'train_total_loss':[],
    'train_recon_loss':[],
    'train_rq_loss':[],
    'train_sids_num':[],
    'train_entropy':[],
    'val_total_loss':[],
    'val_recon_loss':[],
    'val_rq_loss':[],
    'val_sids_num':[],
    'val_entropy':[]}
    
    for e in range(n_ep):
        model.train()
        loss_total_avg = 0 
        loss_recon_avg = 0 
        loss_rq_avg = 0 
        sids_number_avg = torch.zeros(size=(n_layers,), device = device) 
        epoch_sids = [[] for _ in range(n_layers)]
        train_tqdm = tqdm(dl_train, leave = False, desc=f'train {e+1}/{n_ep}') 
        
        for related_id, emb in train_tqdm:
            emb = emb.to(device)
            related_id = related_id.to(device)
            if isinstance(model,RQVAE_modified):
                    pred = model(emb, related_id)
                    q_loss, r_loss, con_loss, loss, sids = pred['q_loss'], pred['r_loss'], pred['con_loss'],pred['loss'], pred['sids']
            else:
                    pred = model(emb)
                    q_loss, r_loss, loss,  sids = pred['q_loss'], pred['r_loss'], pred['loss'], pred['sids']
            loss_recon = r_loss
            un_sids = unique_per_level(sids)
            for level in range(n_layers):
                epoch_sids[level].append(sids[level])

            total_loss = loss
            train_tqdm.set_postfix( total=f"{total_loss.item():.4f}", quant=f"{q_loss.item():.4f}", recon=f"{loss_recon.item():.4f}", uniq = f"{un_sids.tolist()}") 
            
            optimizer.zero_grad() 
            total_loss.backward() 
            optimizer.step()
            
            loss_total_avg += total_loss.item() 
            loss_recon_avg += loss_recon.item() 
            loss_rq_avg += q_loss.item() 
            sids_number_avg += un_sids
            
        loss_total_avg = loss_total_avg/ len(dl_train) 
        loss_recon_avg = loss_recon_avg/ len(dl_train) 
        loss_rq_avg = loss_rq_avg/ len(dl_train) 
        sids_number_avg = sids_number_avg/len(dl_train)
        entropy_levels = entropy_per_level(epoch_sids, codebook_num)

        history['train_total_loss'].append(loss_total_avg) 
        history['train_recon_loss'].append(loss_recon_avg)
        history['train_rq_loss'].append(loss_rq_avg) 
        history['train_sids_num'].append(sids_number_avg.detach().cpu())
        history['train_entropy'].append(entropy_levels)

        model.eval()
        val_tqdm = tqdm(dl_val, leave=False, desc = f'val {e+1}/{n_ep}') 
        loss_total_avg = 0 
        loss_recon_avg = 0 
        loss_rq_avg = 0 
        sids_number_avg = torch.zeros(size=(n_layers,), device = device) 
        epoch_sids = [[] for _ in range(n_layers)]


        with torch.no_grad(): 
            for related_id, emb in val_tqdm: 
                emb = emb.to(device) 
                related_id = related_id.to(device)
                if isinstance(model,RQVAE_modified):
                    pred = model(emb, related_id)
                    q_loss, r_loss, con_loss, loss, sids = pred['q_loss'], pred['r_loss'],  pred['con_loss'],pred['loss'], pred['sids']
                else:
                    pred = model(emb)
                    q_loss, r_loss, loss,  sids = pred['q_loss'], pred['r_loss'], pred['loss'], pred['sids']
                loss_recon = r_loss
                un_sids = unique_per_level(sids)
                for level in range(n_layers):
                    epoch_sids[level].append(sids[level])
                total_loss = loss
                val_tqdm.set_postfix( total=f"{total_loss.item():.4f}", quant=f"{q_loss:.4f}", recon=f"{loss_recon.item():.4f}",uniq = f"{un_sids.tolist()}") 
                
                loss_total_avg += total_loss.item() 
                loss_recon_avg += loss_recon.item()
                loss_rq_avg += q_loss.item()
                sids_number_avg += un_sids
                
            loss_total_avg = loss_total_avg/ len(dl_val) 
            loss_recon_avg = loss_recon_avg/ len(dl_val)
            loss_rq_avg = loss_rq_avg/ len(dl_val)
            sids_number_avg = sids_number_avg/len(dl_val) 
            entropy_levels = entropy_per_level(epoch_sids, codebook_num)

            history['val_total_loss'].append(loss_total_avg) 
            history['val_recon_loss'].append(loss_recon_avg) 
            history['val_rq_loss'].append(loss_rq_avg) 
            history['val_sids_num'].append(sids_number_avg.detach().cpu()) 
            history['val_entropy'].append(entropy_levels) 

            
    return history
        

In [89]:
rq_vae_mod = RQVAE_modified(embeds.shape[1], hidden_sizes, embed_dim, n_layers, beta, gamma).to(device)
optimizer_mod = optim.Adagrad(rq_vae_mod.parameters(), lr=lr)

In [ ]:
for session in range(1, 8):
    cur_hist = train_model(rq_vae_mod, n_ep, optimizer_mod, dl_train_rq, dl_val_rq, n_layers, device)
    print(f"Session {session} has ended!")
    torch.save({
    'history': cur_hist,
    'model_state': rq_vae_mod.state_dict(),
    }, f'checkpoint_{session}.pt')

Session 1 has ended!


Session 2 has ended!


train 624/3000:   0%|          | 0/11 [00:00<?, ?it/s, quant=0.0184, recon=0.1004, total=1.4906, uniq=[68, 50, 51]]           IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)

train 1426/3000:  36%|███▋      | 4/11 [00:00<00:00, 35.19it/s, quant=0.0181, recon=0.0850, total=1.5061, uniq=[68, 54, 51]] IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)



Session 5 has ended!


train 210/3000:  36%|███▋      | 4/11 [00:00<00:00, 34.07it/s, quant=0.0149, recon=0.0802, total=1.2207, uniq=[66, 51, 52]] IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)

train 1465/3000:  73%|███████▎  | 8/11 [00:00<00:00, 35.00it/s, quant=0.0155, recon=0.0791, total=1.3021, uniq=[69, 53, 51]] IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)



Session 6 has ended!


train 419/3000:  73%|███████▎  | 8/11 [00:00<00:00, 34.57it/s, quant=0.1096, recon=0.0987, total=37.6458, uniq=[61, 66, 54]] IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)

train 1335/3000:  73%|███████▎  | 8/11 [00:00<00:00, 34.82it/s, quant=0.1302, recon=0.0980, total=52.5687, uniq=[67, 75, 57]] 

In [59]:
rq_vae = RQVAE(embeds.shape[1], hidden_sizes, embed_dim, codebook_num, n_layers, beta).to(device)
optimizer = optim.Adagrad(rq_vae.parameters(), lr=lr)
rq_vae.load_state_dict(torch.load('/kaggle/input/datasets/qwerte123/best-rqvae/best_rqvae.pth'))

<All keys matched successfully>